# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a Dataset instance, not a dict.

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Citation: {getattr(metadata, 'citeAs', 'N/A')}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate all record sets defined in the schema, and for each one, list the `@id`s of its fields and associated columns. All references will use the canonical `@id` values per the Croissant schema.

In [ ]:
# Display record sets and their fields
print("Available Record Sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id_}")
    print(f"  Name: {getattr(rs, 'name', 'N/A')}")
    print(f"  Description: {getattr(rs, 'description', 'N/A')}")
    print("  Fields and Columns:")
    for field in rs.fields:
        print(f"    Field @id: {field.id_}, Name: {getattr(field, 'name', 'N/A')}, DataType: {getattr(field, 'data_type', 'N/A')}")
        if getattr(field, 'columns', None):
            for col in field.columns:
                print(f"      - Column @id: {col.id_}, Name: {getattr(col, 'name', col.id_)}")
    print()
if not record_sets:
    print("No record sets defined in this dataset schema.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In this example, we autodetect available record sets (the typical dataset has at least one), and load all available data into DataFrames.

In [ ]:
dataframes = {}
for rs in record_sets:
    print(f"Loading records for record set: {rs.id_} ({getattr(rs, 'name', 'N/A')})")
    records = list(dataset.records(record_set=rs.id_))
    df = pd.DataFrame(records)
    dataframes[rs.id_] = df
    print(f"- Columns: {df.columns.tolist()}")
    print(df.head(), "\n")

if not dataframes:
    print("No dataframes built. Dataset may not contain record sets or data extracts.")

## 4. Exploratory Data Analysis (EDA)
We will apply basic analysis: 
* Filter records (for example, on a numeric field),
* Normalize a numeric field,
* Group by a categorical or experimental field.

First, choose a record set and a numeric field from the above overview (using their `@id`).

In [ ]:
# Choose the main record set to use (replace with actual @id from the print above)
if dataframes:
    # Pick the first record set present
    main_record_set_id = list(dataframes.keys())[0]
    df = dataframes[main_record_set_id]

    # Show all columns to decide on fields
    print(f"Available columns in '{main_record_set_id}':\n{df.columns.tolist()}\n")

    # Try to auto-detect a relevant numeric field (e.g., containing 'abundance', 'MW', or 'coverage')
    possible_numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])] + [c for c in df.columns if any(x in c.lower() for x in ['abundance','mw','coverage','count','score','percent'])]
    print(f"Possible numeric fields: {possible_numeric_fields}")

    # We'll use the first detected numeric field which is not all null
    for nf in possible_numeric_fields:
        try:
            df[nf] = pd.to_numeric(df[nf], errors='coerce')
        except Exception:
            continue
    numeric_field_id = None
    for nf in possible_numeric_fields:
        if df[nf].dtype.kind in 'iufc' and df[nf].notnull().sum() > 0:
            numeric_field_id = nf
            break
    if not numeric_field_id:
        print("No usable numeric field detected for EDA.")
    else:
        print(f"Selected numeric field for EDA: {numeric_field_id}\n")

        # Filtering
        threshold = df[numeric_field_id].quantile(0.75) if df[numeric_field_id].notnull().sum() > 0 else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f}, count: {len(filtered_df)}")
        print(filtered_df[[numeric_field_id]].head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a likely experimental group or type column
        possible_group_fields = [c for c in df.columns if any(x in c.lower() for x in ['group', 'experiment', 'condition', 'sample', 'type'])]
        group_field = possible_group_fields[0] if possible_group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"\nGrouped mean {numeric_field_id} by '{group_field}':")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize the distribution of the main numeric field, and if grouping is possible, compare groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered records)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR^2 mass spectrometry human protein dataset with `mlcroissant`, reviewed all data entities by `@id`, and performed basic exploration and visualization using record set and field IDs.

- All entities are referenced by their unique `@id` for robust, reproducible analysis.
- The dataset includes fields for protein identity, abundance, modification, and experiment details, making it suitable for downstream bioinformatics or data science workflows.

**Next steps:** Further domain-specific analysis (e.g., biomarker mining, sequence-based grouping) can easily build on these extracted DataFrames.